# ChromaFit Demonstration Notebook

Welcome to the demonstration notebook for **ChromaFit**, a generic Python library for spectroscopic baseline correction, curve fitting, and peak deconvolution.

This notebook demonstrates how to:
1. Generate or load a UV-Vis absorbance spectrum.
2. Perform baseline subtraction (using polynomial fits or automated Asymmetric Least Squares).
3. Fit overlapping curves using multiple profiles (Gaussian, Lorentzian, Voigt, Pseudo-Voigt, Asymmetric Gaussian).
4. Deconvolute individual peaks, find their centers, and calculate the Area Under the Curve (AUC).
5. Plot and report the results programmatically.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import dyefit as df

print(f"ChromaFit version: {df.__version__}")

## 1. Generate Synthetic Spectrum Data

Let's create a simulated spectrum resembling Methylene Blue, consisting of a quadratic baseline shift, two overlapping peaks (at 612 nm and 664 nm), and some measurement noise.

In [ ]:
# Wavelength range from 400 nm to 800 nm
wavelength = np.arange(400, 800, 1.0)

# Background baseline slope + quadratic curve
baseline_true = 0.03 + 0.0001 * (wavelength - 400) + 0.000002 * (wavelength - 500)**2

# Two simulated absorption peaks
peak1_true = 0.16 * np.exp(-((wavelength - 664.0) / 18.0)**2)  # Monomer peak
peak2_true = 0.08 * np.exp(-((wavelength - 612.0) / 25.0)**2)  # Dimer shoulder

# Add noise
noise = np.random.normal(0, 0.001, len(wavelength))
absorbance_raw = baseline_true + peak1_true + peak2_true + noise

# Plot the simulated spectrum
plt.figure(figsize=(7, 4))
plt.plot(wavelength, absorbance_raw, 'k-', label='Simulated Raw Absorbance')
plt.plot(wavelength, baseline_true, 'r--', label='True Baseline Background')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Absorbance')
plt.title('Synthetic Spectroscopic Data')
plt.legend()
plt.grid(True)
plt.show()

## 2. Baseline Subtraction

ChromaFit supports both manual polynomial fitting and automated Asymmetric Least Squares (ALS) baseline correction. Let's compare both.

In [ ]:
# Retrieve preset parameters for Methylene Blue
config = df.get_dye_config("Methylene Blue")

# Crop spectrum to wavelength range of interest (500 to 750 nm)
crop_mask = (wavelength >= config.min_wavelength) & (wavelength <= config.max_wavelength)
x_data = wavelength[crop_mask]
y_data = absorbance_raw[crop_mask]

# Method A: Polynomial Baseline Fitting (using baseline region mask)
poly_mask = config.baseline_mask_fn(x_data)
y_poly_baseline = df.polynomial_baseline(x_data, y_data, poly_mask, degree=2)
y_corrected_poly = y_data - y_poly_baseline

# Method B: Asymmetric Least Squares (ALS) Subtraction (Automatic - no mask required)
y_als_baseline = df.als_baseline(y_data, lam=1e5, p=0.01)
y_corrected_als = y_data - y_als_baseline

# Plot both baseline corrections side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

# Plot Polynomial Fit
ax1.plot(x_data, y_data, 'gray', label='Raw')
ax1.plot(x_data, y_poly_baseline, 'r--', label='Polynomial Baseline')
ax1.plot(x_data, y_corrected_poly, 'b-', label='Corrected')
ax1.set_title('Polynomial Baseline Correction (Degree 2)')
ax1.set_xlabel('Wavelength (nm)')
ax1.set_ylabel('Absorbance')
ax1.legend()
ax1.grid(True)

# Plot ALS Fit
ax2.plot(x_data, y_data, 'gray', label='Raw')
ax2.plot(x_data, y_als_baseline, 'g--', label='ALS Baseline')
ax2.plot(x_data, y_corrected_als, 'b-', label='Corrected')
ax2.set_title('Asymmetric Least Squares (ALS) Baseline')
ax2.set_xlabel('Wavelength (nm)')
ax2.legend()
ax2.grid(True)

plt.show()

## 3. Automated Peak Detection & Parameter Initialization

ChromaFit uses automated peak finding to estimate center wavelengths and initial guesses. Let's see how it finds the peak positions in our baseline-corrected data.

In [ ]:
# Find peak centers
detected_centers, detected_amps = df.auto_detect_peaks(x_data, y_corrected_poly, num_peaks=2)

print("Detected Peak Wavelengths (nm):", detected_centers)
print("Estimated Absorbances at peaks:", detected_amps)

# Visualize peak locations on the spectrum
plt.figure(figsize=(7, 4))
plt.plot(x_data, y_corrected_poly, 'b-', label='Corrected Spectrum')
plt.scatter(detected_centers, detected_amps, color='red', marker='x', s=100, zorder=5, label='Detected Peaks')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Absorbance')
plt.title('Automated Peak Detection')
plt.legend()
plt.grid(True)
plt.show()

## 4. Multi-Peak Curve Fitting and Deconvolution

Now we'll fit the corrected data using the **Gaussian** model, deconvolute the peaks, and calculate their Area Under the Curve (AUC).

In [ ]:
model_type = "Gaussian"
num_peaks = 2

# Perform curve fitting
popt, pcov, y_fitted, individual_peaks = df.fit_spectrum(
    x_data, y_corrected_poly, model_type, num_peaks=num_peaks
)

# Print goodness of fit statistics
r_squared = df.calculate_r_squared(y_corrected_poly, y_fitted)
rmse = df.calculate_rmse(y_corrected_poly, y_fitted)
print(f"Fit Statistics ({model_type}):")
print(f"  R-squared: {r_squared:.5f}")
print(f"  RMSE:      {rmse:.5f}")
print("-" * 40)

# Report fitted parameters and calculate AUC
num_p = df.get_num_params(model_type)
for i in range(num_peaks):
    peak_params = popt[i * num_p : (i + 1) * num_p]
    peak_y = individual_peaks[i]
    auc = df.calculate_auc(x_data, peak_y)
    
    print(f"Peak {i+1}:")
    print(f"  Amplitude (Height): {peak_params[0]:.4f}")
    print(f"  Center (Wavelength): {peak_params[1]:.2f} nm")
    print(f"  Width Parameter:     {peak_params[2]:.2f}")
    print(f"  Area Under Curve:    {auc:.4f}")
    print("-" * 40)

# Generate the static Matplotlib deconvolution plot
fig = df.generate_matplotlib_fig(x_data, y_corrected_poly, y_fitted, individual_peaks, model_type)
plt.show()

## 5. Fit using Alternative Peak Profiles

Let's repeat the fit using the **Pseudo-Voigt** model, which accounts for both Gaussian and Lorentzian shapes.

In [ ]:
model_type = "Pseudo-Voigt"

popt, pcov, y_fitted, individual_peaks = df.fit_spectrum(
    x_data, y_corrected_poly, model_type, num_peaks=num_peaks
)

r_squared = df.calculate_r_squared(y_corrected_poly, y_fitted)
rmse = df.calculate_rmse(y_corrected_poly, y_fitted)
print(f"Fit Statistics ({model_type}):")
print(f"  R-squared: {r_squared:.5f}")
print(f"  RMSE:      {rmse:.5f}")
print("-" * 40)

num_p = df.get_num_params(model_type)
for i in range(num_peaks):
    peak_params = popt[i * num_p : (i + 1) * num_p]
    peak_y = individual_peaks[i]
    auc = df.calculate_auc(x_data, peak_y)
    
    print(f"Peak {i+1}:")
    print(f"  Amplitude (Height):  {peak_params[0]:.4f}")
    print(f"  Center (Wavelength):  {peak_params[1]:.2f} nm")
    print(f"  Width Parameter:      {peak_params[2]:.2f}")
    print(f"  Lorentzian Fraction:  {peak_params[3]:.2f}")
    print(f"  Area Under Curve:     {auc:.4f}")
    print("-" * 40)

fig = df.generate_matplotlib_fig(x_data, y_corrected_poly, y_fitted, individual_peaks, model_type)
plt.show()